# Text 생성 및 Prompt 예제

### Responses API
| 기능 | **특징** |
| --- | --- |
| **상태 유지(Stateful)** | ✅ 서버 측에서 **대화 상태를 관리(저장 및 이어쓰기 가능)** |
| **내장 도구 지원(Built-in Tools)** | ✅ `web_search`, `file_search`, `computer_use_preview` 등 자동 호출 가능 |
| **구조 & 사용 방식** | `instructions`, `input`, `tools`, `store`, `previous_response_id` 기반 API |
| **응답 형식 (Output)** | `response.output_text`, `response.output` 구조로 단일 메세지 응답 및 도구 호출 결과 포함 |
| **사용 목적** | ✅ **에이전트/도구 사용, 파일/웹 검색, 상태 관리**가 필요한 복잡한 워크플로우에 적합 |
| **미래 확장성** | 🔧 OpenAI Roadmap 중심 – Chat API의 기능을 통합 발전시키는 방향 |유지                                    |

[1]: https://simonwillison.net/2025/Mar/11/responses-vs-chat-completions/?utm_source=chatgpt.com "OpenAI API: Responses vs. Chat Completions"
[2]: https://cookbook.openai.com/examples/responses_api/responses_example?utm_source=chatgpt.com "Web Search and States with Responses API - OpenAI Cookbook"


In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI
client = OpenAI()

Model = "gpt-5-nano"

## Responses API

## 문법 수정

SYSTEM : 당신은 문장을 받게 될 것이며, 당신의 임무는 그것을 표준 한국어로 변환하는 것입니다.  
USER :   
안갔어 시장에 그녀는.

In [3]:
# Responses API 호출 예시
response = client.responses.create(
    model=Model,  
    instructions="당신은 문장을 받게 될 것이며, 당신의 임무는 그것을 표준 한국어로 변환하는 것입니다.",
    input="안갔어 시장에 그녀는."
)

# 응답 결과 출력
print(response.output_text)

그녀는 시장에 가지 않았다.


## 구조화되지 않은 데이터의 구문 분석
SYSTEM : 구조화되지 않은 데이터가 제공되며 이를 CSV 형식으로 구문 분석하는 작업이 수행됩니다.  
USER :   
최근 발견된 행성 구크럭스(Goocrux)에서는 많은 과일이 발견됐다. 그곳에서 자라는 네오스키즐이 있는데, 보라색이고 사탕 맛이 납니다. 회색 빛이 도는 파란색 과일이고 매우 시큼하며 레몬과 약간 비슷한 로헤클(loheckles)도 있습니다. 포유닛은 밝은 녹색을 띠며 단맛보다 풍미가 더 좋습니다. 네온 핑크색 맛과 솜사탕 같은 맛이 나는 루프노바도 많이 있습니다. 마지막으로 글로울(glowls)이라는 과일이 있는데, 이 과일은 신맛과 부식성이 있는 매우 신맛과 쓴맛이 나며 옅은 오렌지색을 띠고 있습니다.

In [4]:
TEXT = """
      최근 발견된 행성 구크럭스(Goocrux)에서는 많은 과일이 발견됐다. 그곳에서 자라는 네오스키즐이 있는데, 보라색이고 사탕 맛이 납니다. 
      회색 빛이 도는 파란색 과일이고 매우 시큼하며 레몬과 약간 비슷한 로헤클(loheckles)도 있습니다. 포유닛은 밝은 녹색을 띠며 단맛보다 풍미가 더 좋습니다. 
      네온 핑크색 맛과 솜사탕 같은 맛이 나는 루프노바도 많이 있습니다. 마지막으로 글로울(glowls)이라는 과일이 있는데, 
      이 과일은 신맛과 부식성이 있는 매우 신맛과 쓴맛이 나며 옅은 오렌지색을 띠고 있습니다.
      이 데이터를 과일명, 색상, 맛으로 분석해 주세요.
      """

response = client.responses.create(
    model=Model,
    instructions="구조화되지 않은 데이터가 제공되며 이를 CSV 형식으로 구문 분석하는 작업이 수행됩니다.",
    input=TEXT
)

print(response.output_text)

과일명,색상,맛
네오스키즐,보라색,사탕 맛
로헤클,회색빛 도는 파란색,매우 시큼하고 레몬 비슷한 맛
포유닛,밝은 녹색,풍미가 강한 맛
루프노바,네온 핑크색,솜사탕 같은 맛
글로울,옅은 오렌지색,매우 신맛과 쓴맛(부식성 있음)


In [5]:
from typing import List
from pydantic import BaseModel

# 개별 과일 정보 모델
class Fruit(BaseModel):
    name: str
    color: str
    taste: str

# 전체 응답 모델
class FruitList(BaseModel):
    fruits: List[Fruit]

response = client.responses.parse(
    model=Model,
    instructions=(
        "구조화되지 않은 문장에서 과일 정보를 추출하세요."
        "반드시 fruits 배열에 각 과일의 name, color, taste를 구조화해서 반환하세요."
    ),
    input=TEXT,
    text_format=FruitList,
)

# 파싱된 구조화 결과
parsed = response.output_parsed

for fruit in parsed.fruits:
    print(dict(fruit))

{'name': '네오스키즐', 'color': '보라색', 'taste': '사탕 맛'}
{'name': '로헤클(loheckles)', 'color': '회색빛이 도는 파란색', 'taste': '매우 시큼하며 레몬과 약간 비슷한'}
{'name': '포유닛', 'color': '밝은 녹색', 'taste': '단맛보다 풍미가 더 좋다'}
{'name': '루프노바', 'color': '네온 핑크색', 'taste': '솜사탕 같은 맛'}
{'name': '글로울(glowls)', 'color': '옅은 오렌지색', 'taste': '매우 신맛과 쓴맛(부식성)'}


## 발화자 관리

| 역할       | 설명                                                                                      |
|------------|-------------------------------------------------------------------------------------------|
| developer  | 애플리케이션 개발자가 제공한 지침. **user 메시지보다 우선적으로 처리**됩니다.              |
| user       | 최종 사용자가 입력한 지침. developer 메시지보다 **우선순위가 낮습니다.**                  |
| assistant  | 모델이 생성한 응답 메시지. 응답 결과는 이 역할로 반환됩니다.                              |


In [6]:
response = client.responses.create(
    model=Model,
    input=[
        {"role": "developer", "content": "사용자에게 한국어로, 초보자도 이해하기 쉽게 5문장 이내로 설명하세요."},
        {"role": "user", "content": "안녕하세요."},
        {"role": "assistant", "content": "안녕하세요. 무었을 도와드릴까요?"},
        {"role": "user", "content": "딥러닝과 머신러닝의 차이점이 무었인가요?"},
    ],
)

print(response.output_text)

머신러닝은 데이터를 이용해 패턴을 찾고 예측하는 일반적인 방법이에요.  
딥러닝은 머신러닝의 한 분야로, 인공신경망을 이용해 더 복잡한 패턴을 자동으로 학습합니다.  
전통적인 머신러닝은 데이터에서 필요한 특징을 사람이 설계해주는 특징 추출이 중요하지만, 딥러닝은 모델이 특징을 자동으로 뽑아줘요.  
딥러닝은 대량의 데이터와 강한 계산 자원이 필요하고, 특히 이미지나 음성 같은 복잡한 데이터에 강력합니다.  
두 방법은 서로 연결되어 있으며, 프로젝트에 따라 어느 쪽이 더 적합한지 선택하면 됩니다.


## Keyword 추출 

In [7]:
text = """
"블랙 온 블랙 도자기(Black-on-Black ware)는 뉴멕시코 북부의 푸에블로 원주민 도자기 예술가들이 개발한 20세기 및 21세기 도자기 전통입니다. 
전통적인 환원 소성 블랙웨어는 푸에블로 예술가들에 의해 수세기 동안 만들어졌습니다. 
지난 세기의 흑색 자기는 표면이 매끄럽고 선택적 버니싱이나 내화 슬립을 적용하여 디자인을 적용한 제품입니다. 
또 다른 스타일은 디자인을 조각하거나 절개하고 융기된 부분을 선택적으로 연마하는 것입니다. 
여러 세대에 걸쳐 Kha'po Owingeh와 P'ohwhóge Owingeh 푸에블로의 여러 가족은 여주인 도예가들로부터 전수받은 기술을 사용하여 검은 바탕에 검은 도자기를 만들어 왔습니다. 
다른 푸에블로 출신의 예술가들도 검정색 바탕에 검정색 도자기를 제작했습니다. 몇몇 현대 예술가들은 조상의 도자기를 기리는 작품을 만들었습니다."
"""

In [8]:
response = client.responses.create(
    model=Model,
    instructions="텍스트 블록이 제공되며, 당신의 임무는 텍스트 블록에서 키워드 목록을 중요한 5 개만 추출하는 것입니다. 한글로 추출합니다.",
    input=text
)

In [9]:
print(response.output_text)

- 블랙 온 블랙 도자기
- 푸에블로 원주민 도자기
- 뉴멕시코 북부
- 환원 소성
- Kha'po Owingeh와 P'ohwhóge Owingeh 푸에블로


## Program Code 설명 

In [10]:
codes = """
class Log:
    def __init__(self, path):
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)
        f = open(path, "a+")

        # Check that the file is newline-terminated
        size = os.path.getsize(path)
        if size > 0:
            f.seek(size - 1)
            end = f.read(1)
            if end != "\n":
                f.write("\n")
        self.f = f
        self.path = path

    def log(self, event):
        event["_event_id"] = str(uuid.uuid4())
        json.dump(event, self.f)
        self.f.write("\n")

    def state(self):
        state = {"complete": set(), "last": None}
        for line in open(self.path):
            event = json.loads(line)
            if event["type"] == "submit" and event["success"]:
                state["complete"].add(event["id"])
                state["last"] = event
        return state
"""

In [11]:
response = client.responses.create(
    model=Model,
    instructions="당신에게는 코드 조각이 제공될 것이며, 무었을 하는 코드인지 한줄로 설명하세요. 한국어로 설명해 주세요.",
    input=codes
)

In [12]:
print(response.output_text)

경로의 디렉토리를 만들고 로그 파일에 JSON 라인씩 기록하며 각 이벤트에 고유 ID를 부여하고, 파일 끝에 개행이 있도록 관리하며, state()에서 완료된 submit 이벤트를 모아 상태를 반환한다.


## 감성 분류기
- 한개의 text 감성 분석

In [13]:
response = client.responses.create(
    model=Model,
    instructions="당신은 텍스트를 입력 받게 될 것이고, 당신의 임무는 텍스트의 감정을 긍정적, 중립적, 부정적으로 분류하는 것입니다.",
    input="나는 새로운 배트맨 영화가 좋습니다!"
)

In [14]:
print(response.output_text)

긍정적


- 여러개의 text 감성 분석

In [15]:
sentiments = [
    "나는 새로운 배트맨 영화가 좋습니다!",
    "오늘 날씨가 너무 안 좋네요.",
    "커피 마시러 가는 길이 설레네요!",
    "회의가 너무 길고 지루했습니다.",
    "점심으로 먹은 피자가 맛있었어요."
]

responses = []
for sent in sentiments:
    response = client.responses.create(
        model=Model,
        instructions="당신은 텍스트를 입력 받게 될 것이고, 당신의 임무는 텍스트의 감정을 긍정적, 중립적, 부정적으로 분류하는 것입니다.",
        input=sent
    )
    # 감정 분석 결과 저장
    responses.append({
        "text": sent,
        "sentiment": response.output_text
    })

# 결과 출력
for result in responses:
    print(f"Sentiments: {result['text']}\nSentiment: {result['sentiment']}\n")

Sentiments: 나는 새로운 배트맨 영화가 좋습니다!
Sentiment: 긍정적

Sentiments: 오늘 날씨가 너무 안 좋네요.
Sentiment: 부정적

Sentiments: 커피 마시러 가는 길이 설레네요!
Sentiment: 긍정적

Sentiments: 회의가 너무 길고 지루했습니다.
Sentiment: negative

Sentiments: 점심으로 먹은 피자가 맛있었어요.
Sentiment: 긍정적



## 실습: 위의 Prompt 내용을 수정해 가며 api 실행